# Test `gemini-2.5-flash-lite` on 14 MMLU-Pro samples

Load the 280-sample CSV, draw 14 random questions, prompt the model, parse the letter choice, and report error rate.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

from llm import chat

MODEL = "gemini-2.5-flash-lite"
CSV_PATH = Path("data/mmlu_pro_sample_20_per_category.csv")
N = 14
SEED = 42

In [2]:
df = pd.read_csv(CSV_PATH)
sample = df.sample(n=N, random_state=SEED).reset_index(drop=True)
sample[["question_id", "category", "answer"]]

,question_id,category,answer
0,10167,physics,D
1,5622,other,B
2,10532,computer science,B
3,10436,computer science,F
4,6501,health,B
5,7763,math,C
6,6604,health,A
7,3445,biology,F
8,6813,health,C
9,6588,health,F


In [3]:
LETTERS = "ABCDEFGHIJ"


def format_prompt(row) -> str:
    options = json.loads(row["options"])
    choices = "\n".join(f"{LETTERS[i]}. {opt}" for i, opt in enumerate(options))
    return (
        f"The following is a multiple choice question about {row['category']}. "
        'Think step by step, then finish with "the answer is (X)" '
        "where X is the correct letter.\n\n"
        f"{row['question']}\n{choices}"
    )


def parse_choice(text: str) -> str | None:
    matches = re.findall(r"(?:the )?answer is\s*\(?([A-J])\)?", text, re.I)
    if matches:
        return matches[-1].upper()
    matches = re.findall(r"\(([A-J])\)", text)
    if matches:
        return matches[-1].upper()
    matches = re.findall(r"\b([A-J])\b", text)
    return matches[-1].upper() if matches else None

In [4]:
rows = []
for i, row in sample.iterrows():
    print(f"[{i + 1}/{N}] {row['category']} id={row['question_id']}")
    output = chat(format_prompt(row), model=MODEL, max_tokens=2048)
    pred = parse_choice(output)
    gold = str(row["answer"]).strip().upper()
    rows.append(
        {
            "question_id": row["question_id"],
            "category": row["category"],
            "gold": gold,
            "predicted": pred,
            "correct": pred == gold,
            "output": output,
        }
    )

results = pd.DataFrame(rows)
results[["question_id", "category", "gold", "predicted", "correct"]]

[1/14] physics id=10167
[2/14] other id=5622
[3/14] computer science id=10532
[4/14] computer science id=10436
[5/14] health id=6501
[6/14] math id=7763
[7/14] health id=6604
[8/14] biology id=3445
[9/14] health id=6813
[10/14] health id=6588
[11/14] philosophy id=10850
[12/14] engineering id=11740
[13/14] philosophy id=11011
[14/14] biology id=3463


,question_id,category,gold,predicted,correct
0,10167,physics,D,I,False
1,5622,other,B,B,True
2,10532,computer science,B,I,False
3,10436,computer science,F,F,True
4,6501,health,B,B,True
5,7763,math,C,C,True
6,6604,health,A,C,False
7,3445,biology,F,D,False
8,6813,health,C,H,False
9,6588,health,F,F,True


In [5]:
n = len(results)
n_correct = int(results["correct"].sum())
n_wrong = n - n_correct
error_rate = n_wrong / n

print(f"Model: {MODEL}")
print(f"Questions: {n}")
print(f"Correct: {n_correct}/{n}")
print(f"Wrong: {n_wrong}/{n}")
print(f"Error rate: {error_rate:.1%}")
print(f"Accuracy: {n_correct / n:.1%}")

Model: gemini-2.5-flash-lite
Questions: 14
Correct: 6/14
Wrong: 8/14
Error rate: 57.1%
Accuracy: 42.9%
